# IBM QPU Implementation Appendix

이 노트북은 Problem 3(b)의 measurement-basis control을 IBM QPU에서 검증하기 위한 appendix용 구현입니다.

중요한 claim boundary:

- 메인 결과는 `solution/Problem 1.ipynb`, `Problem 2.ipynb`, `Problem 3.ipynb`의 state-vector benchmark입니다.
- IBM QPU는 작은 대표 회로가 실제 IBM Runtime 경로로 준비/전송 가능하다는 validation입니다.
- hardware advantage 또는 quantum advantage를 주장하지 않습니다.

## Included QPU Representative Circuits

This notebook directly includes three representative QPU proxy circuits:

- `problem1_random_unitary_one_step`: a tiny 2-qubit smoke-test proxy for the Problem 1 random-unitary scrambling story.
- `hamiltonian_projection_tiny_proxy`: a tiny 3-qubit smoke-test proxy for the Problem 2 Hamiltonian projection story.
- `two_way_postselection_tiny_proxy`: a tiny 3-qubit smoke-test proxy for the Problem 3(c) two-way post-selection story.

These circuits are IBM-compatible representative proxies. They should not be interpreted as full hardware reproductions of Problems 1, 2, and 3. The actual hardware-validation claim remains limited to the Problem 3(b) basis sweep.


## 1. Self-contained IBM Setup

This notebook now contains the IBM validation code directly. It does not import or run `../code/ibm_qpu/*.py`; those scripts are only kept as optional source-code copies.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Self-contained notebook output path. No external IBM script is imported or executed.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "solution":
    PACKAGE_DIR = NOTEBOOK_DIR.parent
else:
    PACKAGE_DIR = Path("C:/Users/koi/Coding/QuantumCylinder/submission/usb_package")

OUTPUT_DIR = PACKAGE_DIR / "code" / "results" / "ibm_qpu_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLAIM_GUARDRAIL = (
    "Appendix validation only. The main scientific claims remain state-vector based; "
    "no hardware advantage or general quantum advantage is claimed."
)

HX = 0.8090
HY = 0.9045
J_COUPLING = 1.0
BETAS = [0.0, math.pi / 4.0, math.pi / 2.0, 3.0 * math.pi / 4.0]
TWO_QUBIT_OPS = {"cx", "cz", "ecr", "rxx"}

print("Notebook directory:", NOTEBOOK_DIR)
print("Package directory:", PACKAGE_DIR)
print("Output directory:", OUTPUT_DIR)


## 2. Representative Circuit Smoke Test Code

This cell directly defines the tiny representative circuits, runtime/backend helpers, transpilation helper, and smoke-test report writer.


In [ ]:
def backend_name(backend: Any | None) -> str | None:
    if backend is None:
        return None
    name = getattr(backend, "name", None)
    if callable(name):
        return str(name())
    if name is not None:
        return str(name)
    return str(backend)


def backend_num_qubits(backend: Any | None) -> int | None:
    if backend is None:
        return None
    num_qubits = getattr(backend, "num_qubits", None)
    if isinstance(num_qubits, int):
        return num_qubits
    configuration = getattr(backend, "configuration", None)
    if callable(configuration):
        try:
            return int(configuration().num_qubits)
        except Exception:
            return None
    return None


def load_runtime_service() -> tuple[Any | None, str, bool]:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
    except Exception as exc:
        return None, f"qiskit-ibm-runtime unavailable: {exc}", False

    token = os.environ.get("QISKIT_IBM_TOKEN") or os.environ.get("IBM_QUANTUM_TOKEN")
    instance = os.environ.get("QISKIT_IBM_INSTANCE")
    kwargs: dict[str, str] = {}
    if token:
        kwargs["token"] = token
    if instance:
        kwargs["instance"] = instance

    try:
        service = QiskitRuntimeService(channel=os.environ.get("QISKIT_IBM_CHANNEL", "ibm_cloud"), **kwargs)
        source = "environment token" if token else "saved Qiskit Runtime account"
        return service, f"loaded from {source}", True
    except TypeError:
        try:
            service = QiskitRuntimeService(**kwargs)
            source = "environment token" if token else "saved Qiskit Runtime account"
            return service, f"loaded from {source}", True
        except Exception as exc:
            return None, f"runtime service unavailable: {exc}", True
    except Exception as exc:
        return None, f"runtime service unavailable: {exc}", True


def select_backend(service: Any | None, backend: str | None, min_qubits: int) -> tuple[Any | None, str]:
    if service is None:
        return None, "no runtime service"
    if backend:
        try:
            return service.backend(backend), f"requested backend {backend}"
        except Exception as exc:
            return None, f"requested backend unavailable: {exc}"
    try:
        return service.least_busy(operational=True, simulator=False, min_num_qubits=min_qubits), "least-busy operational non-simulator backend"
    except Exception as exc:
        return None, f"least-busy backend selection unavailable: {exc}"


def two_qubit_gate_count(circuit: Any) -> int:
    try:
        counts = circuit.count_ops()
        return int(sum(int(counts.get(op, 0)) for op in TWO_QUBIT_OPS))
    except Exception:
        count = 0
        for item in circuit.data:
            operation = getattr(item, "operation", None) or item[0]
            if getattr(operation, "num_qubits", None) == 2:
                count += 1
        return count


def transpile_circuits(circuits: list[Any], backend: Any | None) -> tuple[list[Any], str, str | None]:
    from qiskit import transpile

    if backend is not None:
        try:
            from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
            pass_manager = generate_preset_pass_manager(optimization_level=1, backend=backend)
            return [pass_manager.run(circuit) for circuit in circuits], "preset_pass_manager", None
        except Exception as exc:
            try:
                return list(transpile(circuits, backend=backend, optimization_level=1)), "qiskit_transpile_backend", str(exc)
            except Exception as fallback_exc:
                return circuits, "transpile_failed_returned_original", str(fallback_exc)

    try:
        return list(transpile(circuits, optimization_level=1)), "qiskit_transpile_generic", None
    except Exception as exc:
        return circuits, "generic_transpile_failed_returned_original", str(exc)


def build_representative_circuits(max_circuits: int = 3) -> list[Any]:
    from qiskit import QuantumCircuit

    circuits = []

    qc1 = QuantumCircuit(2, name="problem1_random_unitary_one_step")
    qc1.ry(0.17, 0)
    qc1.rz(-0.23, 1)
    qc1.cx(0, 1)
    qc1.rx(0.31, 0)
    qc1.ry(-0.19, 1)
    qc1.measure_all()
    circuits.append(qc1)

    qc2 = QuantumCircuit(3, name="hamiltonian_projection_tiny_proxy")
    qc2.ry(0.11, 0)
    qc2.rx(-0.08, 1)
    qc2.h(2)
    qc2.cx(0, 2)
    qc2.rz(0.35, 2)
    qc2.cx(0, 2)
    qc2.cx(1, 2)
    qc2.rz(-0.21, 2)
    qc2.cx(1, 2)
    qc2.ry(0.18, 0)
    qc2.measure_all()
    circuits.append(qc2)

    qc3 = QuantumCircuit(3, name="two_way_postselection_tiny_proxy")
    qc3.ry(0.11, 0)
    qc3.rx(-0.08, 1)
    for angle in (0.22, -0.17):
        qc3.h(2)
        qc3.cx(0, 2)
        qc3.rz(angle, 2)
        qc3.cx(0, 2)
        qc3.cx(1, 2)
        qc3.rz(-angle / 2.0, 2)
        qc3.cx(1, 2)
    qc3.measure_all()
    circuits.append(qc3)

    return circuits[:max(1, min(max_circuits, 3))]


def write_smoke_report(report: dict[str, Any], save_dir: Path) -> None:
    save_dir.mkdir(parents=True, exist_ok=True)
    (save_dir / "ibm_qpu_smoke_test_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    lines = [
        "# IBM QPU Smoke Test Summary",
        "",
        f"- dry_run: `{report['dry_run']}`",
        f"- submitted: `{report['submitted']}`",
        f"- backend_selected: `{report['backend_selected']}`",
        f"- transpile_method: `{report['transpile_method']}`",
        "",
        "## Circuits",
        "",
    ]
    for row in report["circuits"]:
        lines.append(f"- `{row['name']}`: qubits={row['num_qubits']}, depth={row['transpiled_depth']}, two_qubit_gates={row['two_qubit_gate_count']}")
    lines.extend(["", CLAIM_GUARDRAIL, ""])
    (save_dir / "ibm_qpu_smoke_test_summary.md").write_text("\n".join(lines), encoding="utf-8")


def run_smoke_dryrun(max_circuits: int = 3, shots: int = 1000, backend: str | None = None) -> dict[str, Any]:
    circuits = build_representative_circuits(max_circuits=max_circuits)
    min_qubits = max(circuit.num_qubits for circuit in circuits)
    service, runtime_status, runtime_available = load_runtime_service()
    selected_backend, backend_selection_status = select_backend(service, backend, min_qubits)
    transpiled, transpile_method, transpile_warning = transpile_circuits(circuits, selected_backend)

    circuit_rows = []
    for original, compiled in zip(circuits, transpiled):
        circuit_rows.append({
            "name": original.name,
            "num_qubits": original.num_qubits,
            "original_depth": original.depth(),
            "transpiled_depth": compiled.depth(),
            "two_qubit_gate_count": two_qubit_gate_count(compiled),
            "operation_counts": dict(compiled.count_ops()),
        })

    report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dry_run": True,
        "submitted": False,
        "shots": shots,
        "requested_backend": backend,
        "backend_selected": backend_name(selected_backend),
        "backend_num_qubits": backend_num_qubits(selected_backend),
        "required_num_qubits": min_qubits,
        "runtime_status": runtime_status,
        "runtime_available": runtime_available,
        "backend_selection_status": backend_selection_status,
        "transpile_method": transpile_method,
        "transpile_warning": transpile_warning,
        "circuits": circuit_rows,
        "claim_guardrail": CLAIM_GUARDRAIL,
    }
    write_smoke_report(report, OUTPUT_DIR / "smoke_test_dryrun")
    return report


## 3. Run Smoke Test Dry-run

This run creates and transpiles representative tiny circuits without submitting an IBM job.


In [ ]:
smoke_report = run_smoke_dryrun(max_circuits=3, shots=1000)
print("dry_run:", smoke_report["dry_run"])
print("submitted:", smoke_report["submitted"])
print("runtime_status:", smoke_report["runtime_status"])
print("backend_selection_status:", smoke_report["backend_selection_status"])
for row in smoke_report["circuits"]:
    print(row["name"], "qubits=", row["num_qubits"], "depth=", row["transpiled_depth"], "two_qubit_gates=", row["two_qubit_gate_count"])


## 3-1. Inspect Representative Circuit Names

This cell prints the names and small resource diagnostics for the three representative QPU proxy circuits used in the smoke test.


In [ ]:
representative_circuits = build_representative_circuits(max_circuits=3)
for circuit in representative_circuits:
    print(circuit.name, 'qubits=', circuit.num_qubits, 'depth=', circuit.depth(), 'two_qubit_gates=', two_qubit_gate_count(circuit))


## 4. Problem 3(b) Basis Sweep Code

This cell directly defines the measurement-basis sweep circuits for the Problem 3(b) appendix validation.


In [ ]:
def build_problem3b_basis_circuits(repeats: int = 1, trotter_steps: int = 1, dt: float = 0.20) -> tuple[list[Any], list[dict[str, Any]]]:
    from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
    from qiskit.circuit.library import RXXGate

    circuits = []
    metadata_rows = []
    index = 0
    for repeat in range(max(1, repeats)):
        for beta in BETAS:
            qreg = QuantumRegister(3, "q")
            creg = ClassicalRegister(3, "meas")
            beta_over_pi = beta / math.pi
            qc = QuantumCircuit(qreg, creg, name=f"p3b_beta_{beta_over_pi:.2f}pi_rep_{repeat}")

            qc.ry(0.045 + 0.010 * repeat, qreg[0])
            qc.rx(-0.035 + 0.006 * repeat, qreg[1])
            qc.rz(0.020 * (repeat + 1), qreg[0])
            qc.ry(-0.015 * (repeat + 1), qreg[1])

            for _ in range(max(1, trotter_steps)):
                for q in range(3):
                    qc.rx(2.0 * HX * dt, qreg[q])
                    qc.ry(2.0 * HY * dt, qreg[q])
                qc.append(RXXGate(2.0 * J_COUPLING * dt), [qreg[0], qreg[1]])
                qc.append(RXXGate(2.0 * J_COUPLING * dt), [qreg[1], qreg[2]])

            qc.ry(-beta, qreg[2])
            qc.measure(qreg[0], creg[0])
            qc.measure(qreg[1], creg[1])
            qc.measure(qreg[2], creg[2])

            circuits.append(qc)
            metadata_rows.append({
                "index": index,
                "circuit_name": qc.name,
                "beta": beta,
                "beta_over_pi": beta_over_pi,
                "repeat": repeat,
            })
            index += 1

    return circuits, metadata_rows


def circuit_rows(original_circuits: list[Any], transpiled_circuits: list[Any], metadata_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for original, compiled, metadata in zip(original_circuits, transpiled_circuits, metadata_rows):
        rows.append({
            **metadata,
            "original_depth": original.depth(),
            "transpiled_depth": compiled.depth(),
            "two_qubit_gate_count": two_qubit_gate_count(compiled),
            "transpiled_ops": dict(compiled.count_ops()),
        })
    return rows


def write_basis_outputs(report: dict[str, Any], rows: list[dict[str, Any]], save_dir: Path) -> None:
    save_dir.mkdir(parents=True, exist_ok=True)
    (save_dir / "problem3b_ibm_basis_sweep_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

    csv_path = save_dir / "problem3b_ibm_basis_sweep_circuits.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["index", "circuit_name", "beta", "beta_over_pi", "repeat", "original_depth", "transpiled_depth", "two_qubit_gate_count", "transpiled_ops"])
        writer.writeheader()
        writer.writerows(rows)

    grouped: dict[float, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        grouped[float(row["beta_over_pi"])].append(row)

    lines = [
        "# Problem 3-b IBM Basis Sweep Dry-run Summary",
        "",
        f"- dry_run: `{report['dry_run']}`",
        f"- submitted: `{report['submitted']}`",
        f"- backend_selected: `{report['backend_selected']}`",
        f"- num_circuits: `{len(rows)}`",
        "",
        "| beta/pi | circuits | mean depth | mean two-qubit gates |",
        "|---:|---:|---:|---:|",
    ]
    for beta, beta_rows in sorted(grouped.items()):
        mean_depth = sum(float(row["transpiled_depth"]) for row in beta_rows) / len(beta_rows)
        mean_twoq = sum(float(row["two_qubit_gate_count"]) for row in beta_rows) / len(beta_rows)
        lines.append(f"| {beta:.2f} | {len(beta_rows)} | {mean_depth:.2f} | {mean_twoq:.2f} |")
    lines.extend(["", CLAIM_GUARDRAIL, ""])
    (save_dir / "problem3b_ibm_basis_sweep_summary.md").write_text("\n".join(lines), encoding="utf-8")


def run_problem3b_basis_sweep_dryrun(repeats: int = 1, shots: int = 256, trotter_steps: int = 1, dt: float = 0.20, backend: str | None = None) -> dict[str, Any]:
    circuits, metadata_rows = build_problem3b_basis_circuits(repeats=repeats, trotter_steps=trotter_steps, dt=dt)
    service, runtime_status, runtime_available = load_runtime_service()
    selected_backend, backend_selection_status = select_backend(service, backend, min_qubits=3)
    transpiled, transpile_method, transpile_warning = transpile_circuits(circuits, selected_backend)
    rows = circuit_rows(circuits, transpiled, metadata_rows)

    report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dry_run": True,
        "submitted": False,
        "retrieve_job": False,
        "shots": shots,
        "repeats": repeats,
        "trotter_steps": trotter_steps,
        "dt": dt,
        "requested_backend": backend,
        "backend_selected": backend_name(selected_backend),
        "backend_num_qubits": backend_num_qubits(selected_backend),
        "runtime_status": runtime_status,
        "runtime_available": runtime_available,
        "backend_selection_status": backend_selection_status,
        "transpile_method": transpile_method,
        "transpile_warning": transpile_warning,
        "num_circuits": len(rows),
        "claim_guardrail": CLAIM_GUARDRAIL,
    }
    write_basis_outputs(report, rows, OUTPUT_DIR / "problem3b_basis_sweep_dryrun")
    return report


## 5. Run Problem 3(b) Basis Sweep Dry-run

This run builds the basis-angle circuits and records depth/two-qubit-gate diagnostics without submitting an IBM job.


In [ ]:
basis_report = run_problem3b_basis_sweep_dryrun(repeats=1, shots=256, trotter_steps=1, dt=0.20)
print("dry_run:", basis_report["dry_run"])
print("submitted:", basis_report["submitted"])
print("backend_selected:", basis_report["backend_selected"])
print("num_circuits:", basis_report["num_circuits"])
print("summary:", OUTPUT_DIR / "problem3b_basis_sweep_dryrun" / "problem3b_ibm_basis_sweep_summary.md")


## 4. Problem 3(b) Basis Sweep Dry-run

이 셀은 Problem 3(b)의 basis angle 후보를 tiny circuit으로 구성합니다.

기본값에서는 `--submit`을 넣지 않으므로 실제 IBM job은 제출되지 않습니다. 검토 목적은 다음입니다.

- basis angle별 회로가 생성되는지 확인
- transpile 결과 depth/gate count가 기록되는지 확인
- appendix validation용 결과 파일 구조가 만들어지는지 확인


In [ ]:
summary_path = OUTPUT_DIR / "problem3b_basis_sweep_dryrun" / "problem3b_ibm_basis_sweep_summary.md"
circuit_csv_path = OUTPUT_DIR / "problem3b_basis_sweep_dryrun" / "problem3b_ibm_basis_sweep_circuits.csv"
report_path = OUTPUT_DIR / "problem3b_basis_sweep_dryrun" / "problem3b_ibm_basis_sweep_report.json"

print("summary exists:", summary_path.exists())
print("circuit csv exists:", circuit_csv_path.exists())
print("report json exists:", report_path.exists())

if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8")[:1200])


## 5. Read Problem 3(b) Basis Sweep Output

basis sweep dry-run은 CSV/JSON/Markdown 결과를 만듭니다.

심사자가 볼 핵심은 `problem3b_ibm_basis_sweep_summary.md`와 `problem3b_ibm_basis_sweep_circuits.csv`입니다.


In [ ]:
# Optional real IBM QPU submission sketch.
# This cell is intentionally non-executing. A real run must explicitly set credentials and call IBM Runtime.
#
# Required environment variables:
# - QISKIT_IBM_TOKEN or IBM_QUANTUM_TOKEN
# - QISKIT_IBM_INSTANCE
#
# The dry-run cells above already build and transpile the circuits directly in this notebook.
# To submit, reuse `build_problem3b_basis_circuits(...)`, select a backend with `load_runtime_service()`
# and `select_backend(...)`, then create a Qiskit Runtime SamplerV2 job with the transpiled circuits.
# Keep this as appendix validation only; do not turn it into a hardware-advantage claim.


## 6. Optional Real IBM QPU Submission

아래 셀은 의도적으로 주석 처리되어 있습니다.

실제 제출 전 필요 조건:

- `QISKIT_IBM_TOKEN` 또는 `IBM_QUANTUM_TOKEN`
- `QISKIT_IBM_INSTANCE`
- 사용할 backend 이름 확인
- job 제출 비용/queue time 확인

실제 제출을 하려면 `--submit`을 명시해야 합니다.
